# SteelDefect: Steel Surface Defect Classification

**arXiv ref:** *Grad-CAM: Visual Explanations from Deep Networks* — ICCV 2017

This notebook demonstrates defect classification on steel surface images using **ResNet18** with Grad-CAM explainability.


In [ ]:
from __future__ import annotations
import sys, os; sys.path.insert(0, os.path.abspath('.'))
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.data import make_synthetic, create_data_splits, DEFECT_CLASSES, N_CLASSES
from src.model import build_resnet18, build_custom_cnn, compute_gradcam
from src.model import evaluate_model


In [ ]:
# Generate synthetic steel defect data
data = make_synthetic(n_per_class=20, img_size=64)
print(f"Samples: {data['n_samples']}")
print(f"Defect classes: {data['class_names']}")
print(f"Class distribution: {data['class_counts']}")


# Visualize defect types
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for i, ax in enumerate(axes.flat):
    mask = data['labels'] == i
    idx = np.where(mask)[0][0]
    img = data['images'][idx].transpose(1, 2, 0)
    img = (img - img.min()) / (img.max() - img.min())
    ax.imshow(img); ax.set_title(DEFECT_CLASSES[i]); ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# Build ResNet18
model = build_resnet18(num_classes=N_CLASSES, pretrained=False)
print(f"ResNet18: {sum(p.numel() for p in model.parameters()):,} params")


In [ ]:
# Forward pass
with torch.no_grad():
    x = torch.FloatTensor(data['images'][:4])
    out = model(x)
print(f"Output: {out.shape}")
print(f"Predictions: {out.argmax(dim=1)}")


In [ ]:
# Grad-CAM explanation
model.train()  # need gradients
x = data['images'][0:1]
result = compute_gradcam(model, x, target_class=0)
print(f"Predicted class: {result['prediction']} ({DEFECT_CLASSES[result['prediction']]})")
print(f"Confidence: {result['confidence']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
img = (x[0].transpose(1, 2, 0) - x[0].min()) / (x[0].max() - x[0].min())
axes[0].imshow(img); axes[0].set_title('Input'); axes[0].axis('off')
axes[1].imshow(img)
axes[1].imshow(result['heatmap'], cmap='jet', alpha=0.5)
axes[1].set_title('Grad-CAM'); axes[1].axis('off')
plt.tight_layout(); plt.show()


In [ ]:
print('SteelDefect notebook complete. Grad-CAM highlights regions that drive the model decision.')
